# Module 22 — Credit Risk Scoring: GBM for Default Probability

> **Track 2 · Revolut Fintech Specialization**  
> **Difficulty**: Intermediate → Advanced  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Scikit-learn, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 10 (Sklearn Pipeline), Module 13 (Logistic Regression)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Loan Application Data](#3-synthetic-loan-application-data)
4. [Feature Engineering](#4-feature-engineering)
5. [Model Training](#5-model-training)
6. [Model Evaluation](#6-model-evaluation)
7. [Probability Calibration](#7-probability-calibration)
8. [Visualization](#8-visualization)
9. [Key Business Takeaway](#9-key-business-takeaway)

---
## §1 · Business Context

### Why credit risk scoring matters?

Credit risk modeling is **core** to lending profitability:
- **Default prediction**: Estimate probability of borrower default.
- **Risk-based pricing**: Charge higher rates for riskier borrowers.
- **Portfolio management**: Maintain healthy risk distribution.

**Key metrics**:
1. **Gini coefficient**: Measures discriminatory power.
2. **KS statistic**: Maximum separation between distributions.
3. **Brier score**: Probability calibration quality.

**Regulatory requirements**:
- **Basel III**: Capital adequacy based on risk models.
- **IFRS 9**: Expected credit loss provisioning.
- **Model validation**: Regular backtesting and monitoring.

In this notebook we will:
1. Generate synthetic loan application data.
2. Engineer credit risk features.
3. Train a GBM model for default probability.
4. Calibrate probabilities and evaluate performance.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, confusion_matrix
)
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Loan Application Data

In [ ]:
# Generate synthetic loan application data
N_APPLICATIONS = 100000
DEFAULT_RATE = 0.08  # 8% default rate

# Generate features
np.random.seed(42)

data = {
    'age': np.random.normal(35, 10, N_APPLICATIONS).clip(18, 70).astype(int),
    'income': np.random.lognormal(10.5, 0.8, N_APPLICATIONS),
    'employment_years': np.random.exponential(5, N_APPLICATIONS).clip(0, 40),
    'debt_to_income': np.random.beta(2, 5, N_APPLICATIONS),
    'credit_score': np.random.normal(650, 100, N_APPLICATIONS).clip(300, 850).astype(int),
    'num_credit_lines': np.random.poisson(3, N_APPLICATIONS),
    'recent_inquiries': np.random.poisson(1, N_APPLICATIONS),
    'loan_amount': np.random.lognormal(9.5, 0.7, N_APPLICATIONS),
    'loan_term': np.random.choice([12, 24, 36, 48, 60], N_APPLICATIONS),
    'has_mortgage': np.random.binomial(1, 0.3, N_APPLICATIONS),
    'has_previous_default': np.random.binomial(1, 0.05, N_APPLICATIONS),
}

df = pd.DataFrame(data)

# Generate default labels (correlated with features)
default_prob = (
    - 0.5
    + 0.3 * (df['credit_score'] - 650) / 100
    - 0.2 * df['debt_to_income']
    + 0.1 * df['has_previous_default']
    - 0.05 * df['employment_years'] / 5
    + np.random.normal(0, 0.1, N_APPLICATIONS)
)
default_prob = 1 / (1 + np.exp(-default_prob))  # Sigmoid
df['default'] = np.random.binomial(1, default_prob)

print(f"✓ Generated loan application data:")
print(f"  Applications: {len(df):,}")
print(f"  Default rate: {df['default'].mean()*100:.2f}%")
print(f"\nFeature statistics:")
print(df.describe().round(2))

---
## §4 · Feature Engineering

In [ ]:
# Create additional features
df['loan_to_income'] = df['loan_amount'] / df['income']
df['monthly_payment'] = df['loan_amount'] / df['loan_term']
df['payment_to_income'] = df['monthly_payment'] / (df['income'] / 12)
df['credit_utilization'] = df['num_credit_lines'] * df['debt_to_income']

# Interaction features
df['credit_income_interaction'] = df['credit_score'] * df['income'] / 100000
df['dti_employment'] = df['debt_to_income'] * df['employment_years']

feature_cols = ['age', 'income', 'employment_years', 'debt_to_income', 'credit_score',
                'num_credit_lines', 'recent_inquiries', 'loan_amount', 'loan_term',
                'has_mortgage', 'has_previous_default', 'loan_to_income',
                'monthly_payment', 'payment_to_income', 'credit_utilization',
                'credit_income_interaction', 'dti_employment']

print(f"✓ Created {len(feature_cols)} features")

---
## §5 · Model Training

In [ ]:
# Train GBM model
X = df[feature_cols]
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)

print("Training credit risk model...")
t0 = time.time()
model.fit(X_train_scaled, y_train)
train_time = time.time() - t0

print(f"✓ Trained in {train_time:.2f}s")

---
## §6 · Model Evaluation

In [ ]:
# Evaluate model
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("=" * 70)
print("CREDIT RISK MODEL EVALUATION")
print("=" * 70)
print(f"\nROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"Brier Score: {brier:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

---
## §7 · Probability Calibration

In [ ]:
# Calibrate probabilities using Platt scaling
calibrated_model = CalibratedClassifierCV(model, method='sigmoid', cv=5)
calibrated_model.fit(X_train_scaled, y_train)

y_prob_calibrated = calibrated_model.predict_proba(X_test_scaled)[:, 1]
brier_calibrated = brier_score_loss(y_test, y_prob_calibrated)

print(f"\nCalibration improvement:")
print(f"  Brier score (before): {brier:.4f}")
print(f"  Brier score (after):  {brier_calibrated:.4f}")
print(f"  Improvement: {(brier - brier_calibrated)/brier*100:.2f}%")

---
## §8 · Visualization

In [ ]:
# Feature importance
importance = model.feature_importances_
feat_imp = pd.DataFrame({'Feature': feature_cols, 'Importance': importance}).sort_values('Importance', ascending=False)

fig = make_subplots(rows=1, cols=2, subplot_titles=['Feature Importance', 'Probability Distribution'])

# Feature importance
fig.add_trace(go.Bar(
    x=feat_imp['Importance'].head(10),
    y=feat_imp['Feature'].head(10),
    orientation='h',
    marker_color='#636EFA'
), row=1, col=1)

# Probability distribution by default status
fig.add_trace(go.Histogram(
    x=y_prob[y_test == 0],
    name='No Default',
    opacity=0.7,
    nbinsx=50
), row=1, col=2)

fig.add_trace(go.Histogram(
    x=y_prob[y_test == 1],
    name='Default',
    opacity=0.7,
    nbinsx=50
), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - Credit score is the top predictor")
print("   - Debt-to-income ratio matters significantly")
print("   - Calibrated probabilities enable better risk pricing")

---
## §9 · Key Business Takeaway

> ### 🎯 Action Items for Credit Risk Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Loan origination, credit line increases, portfolio monitoring |
> | **Features** | Credit score, DTI, employment history, loan characteristics |
> | **Model** | GBM with probability calibration |
> | **Evaluation** | ROC-AUC, Brier score, Gini coefficient |
> | **Deployment** | Real-time scoring with calibrated probabilities |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Risk-based pricing**:
>    ```
>    PD (Probability of Default) → Risk Grade → Interest Rate
>    ```
>
> 2. **Risk grade mapping**:
>    | PD Range | Risk Grade | Interest Rate |
>    |----------|------------|---------------|
>    | 0-2% | A (Low) | 5-8% |
>    | 2-5% | B (Medium) | 8-12% |
>    | 5-10% | C (High) | 12-18% |
>    | >10% | D (Very High) | Decline |
>
> 3. **Model monitoring**:
>    - Track PSI for input features monthly
>    - Monitor actual default rates vs. predicted
>    - Retrain quarterly or when drift detected
>
> ### 🔑 Key Takeaways
>
> 1. **Probability calibration is critical** for risk-based pricing.
> 2. **Credit score and DTI are top predictors** across most models.
> 3. **GBM models outperform logistic regression** for credit risk.
> 4. **Regular backtesting** is required by regulators.
> 5. **Combine ML with expert rules** for robust decisioning.